# Task A: Property type classification (four labels)
Train **TF-IDF + Logistic Regression** and **TF-IDF + Linear SVM**. Tune C on validation macro-F1, evaluate on the untouched test split, save the best model.

The dataset must contain `text` and `label`. Task A needs real labeled listing descriptions; its template is not training data. Task B uses generated synthetic text, so results do not demonstrate performance on real buyers.

In [ ]:
from pathlib import Path
import os, re, unicodedata, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, ConfusionMatrixDisplay
import joblib
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SEED=42

def clean(text):
    text=unicodedata.normalize('NFKC',str(text))
    text=re.sub(r'https?://\S+|www\.\S+',' URL ',text)
    text=re.sub(r'[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}',' EMAIL ',text)
    text=re.sub(r'(?:\+63|0)9\d{9}\b',' PHONE ',text)
    text=re.sub(r'₱\s*[\d,]+(?:\.\d+)?',' PRICE ',text)
    text=re.sub(r'\s+',' ',text).strip()
    return text.lower() # preserve negation: hindi, ayaw, wag, not

def split_data(df):
    trainval,test=train_test_split(df,test_size=.15,random_state=SEED,stratify=df.label)
    train,val=train_test_split(trainval,test_size=.15/.85,random_state=SEED,stratify=trainval.label)
    return train,val,test

def evaluate(y_true,y_pred):
    p,r,f,_=precision_recall_fscore_support(y_true,y_pred,average='macro',zero_division=0)
    return dict(accuracy=accuracy_score(y_true,y_pred),macro_precision=p,macro_recall=r,macro_f1=f)


In [ ]:
data_path=ROOT/'dataset'/'task_a_listings.csv'
if not data_path.exists():
    raise FileNotFoundError(f"Add a labeled CSV at {data_path}. Use the template as a guide.")
df=pd.read_csv(data_path).dropna(subset=['text','label'])
df['text']=df['text'].map(clean)
df=df.drop_duplicates(subset='text').reset_index(drop=True)
print(df.label.value_counts())
train,val,test=split_data(df)
print('train / val / test:',len(train),len(val),len(test))

In [ ]:
results=[]; best=None; best_score=-1
for name, estimator in [('Logistic Regression',LogisticRegression(max_iter=2000,class_weight='balanced')),('Linear SVM',LinearSVC(class_weight='balanced'))]:
    for C in [0.1,1,10]:
        estimator.set_params(C=C)
        pipe=Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True,min_df=2,max_features=50000)),('clf',estimator)])
        pipe.fit(train.text,train.label)
        score=evaluate(val.label,pipe.predict(val.text))['macro_f1']
        results.append({'model':name,'C':C,'val_macro_f1':score})
        if score>best_score: best_score=score;best=pipe;best_name=name;best_C=C
pd.DataFrame(results).sort_values('val_macro_f1',ascending=False)

In [ ]:
# Report BOTH model families on test using validation-selected C per family.
summary=[];predictions={}
for name,estimator in [('Logistic Regression',LogisticRegression(max_iter=2000,class_weight='balanced')),('Linear SVM',LinearSVC(class_weight='balanced'))]:
    candidates=[r for r in results if r['model']==name]
    C=max(candidates,key=lambda r:r['val_macro_f1'])['C']
    estimator.set_params(C=C)
    pipe=Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True,min_df=2,max_features=50000)),('clf',estimator)])
    pipe.fit(pd.concat([train.text,val.text]),pd.concat([train.label,val.label]))
    pred=pipe.predict(test.text); predictions[name]=(pipe,pred)
    summary.append({'model':name,'C':C,**evaluate(test.label,pred)})
metrics=pd.DataFrame(summary);display(metrics)
selected=metrics.sort_values('macro_f1',ascending=False).iloc[0]['model']
chosen,pred=predictions[selected]
joblib.dump(chosen,ROOT/'model'/'listing_best.joblib')
print('Saved:',selected)
print(classification_report(test.label,pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(test.label,pred,xticks_rotation=45);plt.show()
errors=test.copy();errors['predicted']=pred
errors=errors[errors.label!=errors.predicted]
errors.to_csv(ROOT/'results'/'task_a_errors.csv',index=False)
metrics.to_csv(ROOT/'results'/'task_a_baseline_metrics.csv',index=False)
print('Misclassifications:',len(errors))

**Important:** Comparing model performance using the test set to select a winner is exploratory. For a strict final report, choose the deployment model using validation only, then evaluate it once on test. Avoid claiming synthetic-data test scores generalize to real users.